In [1]:
import tensorflow as tf
import numpy as np
import os
from PIL import Image

print(f"TensorFlow Version: {tf.__version__}")

# ==========================================
# STEP 1: CREATE DIRECTORY STRUCTURE
# ==========================================
base_dir = '/content/gatekeeper_dataset'
knee_dir = os.path.join(base_dir, 'knee')
not_knee_dir = os.path.join(base_dir, 'not_knee')

os.makedirs(knee_dir, exist_ok=True)
os.makedirs(not_knee_dir, exist_ok=True)

print("✅ Directory structure created.")
print(f"👉 ACTION REQUIRED: Upload about 300-500 of your Knee X-rays into the '{knee_dir}' folder using the Colab file explorer on the left.")

TensorFlow Version: 2.19.0
✅ Directory structure created.
👉 ACTION REQUIRED: Upload about 300-500 of your Knee X-rays into the '/content/gatekeeper_dataset/knee' folder using the Colab file explorer on the left.


In [2]:
import os
import shutil

# Paths
source_base = '/content/drive/MyDrive/FYP/merged_data'
dest_knee_dir = '/content/gatekeeper_dataset/knee'

# Ensure the destination folder exists
os.makedirs(dest_knee_dir, exist_ok=True)

# We want a balanced dataset: 100 images per KL grade = 500 total 'knee' images
IMAGES_PER_CLASS = 100
total_copied = 0

print("Starting to copy images from Google Drive...\n")

for kl_grade in ['0', '1', '2', '3', '4']:
    source_folder = os.path.join(source_base, kl_grade)

    # Check if the folder exists
    if not os.path.exists(source_folder):
        print(f"⚠️ Warning: Folder not found -> {source_folder}")
        continue

    # Get all valid image files
    files = os.listdir(source_folder)
    image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if len(image_files) == 0:
        print(f"⚠️ Warning: No images found in {source_folder}")
        continue

    # Take a subset of images
    images_to_copy = image_files[:IMAGES_PER_CLASS]

    for img_name in images_to_copy:
        src_path = os.path.join(source_folder, img_name)
        # Prefix the destination filename with the grade to avoid name collisions (e.g., if two folders have "image_1.jpg")
        dest_name = f"grade{kl_grade}_{img_name}"
        dest_path = os.path.join(dest_knee_dir, dest_name)

        shutil.copy2(src_path, dest_path)
        total_copied += 1

    print(f"✅ Copied {len(images_to_copy)} images from KL grade {kl_grade}.")

print(f"\n🎉 Done! Total knee X-rays copied: {total_copied}")
print("You can now proceed to Step 3 in the main script to start training!")

Starting to copy images from Google Drive...

✅ Copied 100 images from KL grade 0.
✅ Copied 100 images from KL grade 1.
✅ Copied 100 images from KL grade 2.
✅ Copied 100 images from KL grade 3.
✅ Copied 100 images from KL grade 4.

🎉 Done! Total knee X-rays copied: 500
You can now proceed to Step 3 in the main script to start training!


In [3]:
!pip install -q medmnist

import os
import medmnist
from medmnist import INFO
from PIL import Image
import numpy as np

not_knee_dir = '/content/gatekeeper_dataset/not_knee'
os.makedirs(not_knee_dir, exist_ok=True)

print("Downloading Medical 'Near-OOD' images...\n")

# We will grab Chest X-Rays and Abdominal/Head CTs to confuse the model
datasets_to_use = ['pneumoniamnist', 'organamnist']
images_per_dataset = 250
total_medical_added = 0

for data_flag in datasets_to_use:
    print(f"Fetching {data_flag}...")
    info = INFO[data_flag]
    DataClass = getattr(medmnist, info['python_class'])

    # Download the dataset
    dataset = DataClass(split='train', download=True)

    # Save the first 250 images from each dataset
    for i in range(min(images_per_dataset, len(dataset))):
        img, _ = dataset[i] # img is a PIL Image
        img = img.resize((256, 256))

        # Save to our not_knee folder
        save_path = os.path.join(not_knee_dir, f"{data_flag}_{i}.jpg")
        img.save(save_path)
        total_medical_added += 1

print(f"\n✅ Success! Added {total_medical_added} confusing medical scans to the 'Not Knee' folder.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.3 MB/s eta 0:00:00

Fetching pneumoniamnist...


100%|██████████| 4.17M/4.17M [00:06<00:00, 673kB/s]


Fetching organamnist...


100%|██████████| 38.2M/38.2M [00:53<00:00, 713kB/s]



✅ Success! Added 500 confusing medical scans to the 'Not Knee' folder.


In [4]:
print("Downloading 'Not Knee' background images...")
(x_train, _), (_, _) = tf.keras.datasets.cifar10.load_data()

# Save 500 random images to the not_knee folder
for i in range(500):
    img_array = x_train[i]
    img = Image.fromarray(img_array)
    img = img.resize((256, 256)) # Resize to match your pipeline
    img.save(os.path.join(not_knee_dir, f'random_ood_{i}.jpg'))

print(f"✅ Generated 500 'Not Knee' images in {not_knee_dir}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
✅ Generated 500 'Not Knee' images in /content/gatekeeper_dataset/not_knee


In [6]:
base_dir = '/content/gatekeeper_dataset'
BATCH_SIZE = 32
IMG_SIZE = (256, 256)

print("Loading datasets into memory...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=0.2,
    subset="training",
    seed=42
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=42
)

# Optimize memory usage during training
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

# ==========================================
# 2. BUILD THE NEURAL NETWORK
# ==========================================
print("\nBuilding Transfer Learning Model (MobileNetV2)...")

# Data augmentation to prevent overfitting
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.1),
])

# Load pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze the base model

# Build our custom classification head
inputs = tf.keras.Input(shape=(256, 256, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)

# Binary classification: 1 output node with sigmoid activation (0.0 to 1.0)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# ==========================================
# 3. TRAIN AND EXPORT
# ==========================================
EPOCHS = 20

print(f"\nStarting training for up to {EPOCHS} epochs with Early Stopping...")

# 1. Define the Early Stopping callback
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',         # Watch the validation loss
    patience=3,                 # Stop if no improvement after 3 epochs
    restore_best_weights=True,  # Automatically keep the best version of the model
    verbose=1
)

# 2. Train the model
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=validation_dataset,
    callbacks=[early_stopping]    # Pass the callback here
)

export_path = '/content/drive/MyDrive/FYP/gatekeeper.keras'
model.save(export_path)
print(f"\n✅ Training complete! Model saved to {export_path}")

Loading datasets into memory...
Found 1500 files belonging to 2 classes.
Using 1200 files for training.
Found 1500 files belonging to 2 classes.
Using 300 files for validation.

Building Transfer Learning Model (MobileNetV2)...


/tmp/ipykernel_3012/3012467674.py:43: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(



Starting training for up to 20 epochs with Early Stopping...
Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 9s 126ms/step - accuracy: 0.9317 - loss: 0.1935 - val_accuracy: 1.0000 - val_loss: 0.0145
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step - accuracy: 1.0000 - loss: 0.0126 - val_accuracy: 1.0000 - val_loss: 0.0066
Epoch 3/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 87ms/step - accuracy: 1.0000 - loss: 0.0084 - val_accuracy: 1.0000 - val_loss: 0.0044
Epoch 4/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 1.0000 - loss: 0.0054 - val_accuracy: 1.0000 - val_loss: 0.0032
Epoch 5/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 1.0000 - loss: 0.0042 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 6/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 95ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 7/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 8/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 88ms/st